# PV Calculator Algorithm
Fahim Usshihab

Grouping Guide <br>
Group A= Control <br>
Group B= REBOA <br>
Group C= EVAC

## User Inputs

In [ ]:
choose_file=1
normal_files=['A 1822.xlsx', #0
              'A 1825.xlsx', #1
              'A 1839.xlsx', #2
              
              'A 1841.xlsx', #3
              'A 1843.xlsx', #4
              'A 1849.xlsx', #5
              
              'B 1809.xlsx', #6
              'B 1827.xlsx', #7
              'B 1832.xlsx', #8
              
              'B 1863.xlsx', #9
              'B 1876.xlsx', #10
              'B 1884.xlsx', #11
              
              'E 1824.xlsx', #12
              'E 1844.xlsx', #13
              'E 1848.xlsx', #14
              
              'E 1882.xlsx', #15
              'E 1883.xlsx', #16
              'E 1813.xlsx'] #17


choose_time_point=3

time_points=['T0.0', #0
            'T0.1',  #1
            'T0.2',  #2
             
            'T30.0', #3
            'T30.1', #4
            'T30.2', #5
             
            'T74.0', #6
            'T74.1', #7
            'T74.2'] #8


time_step=0.01

pressure_index=0
volume_index=1

file_name=normal_files[choose_file]

sheet_name=time_points[choose_time_point]

#file_name='E 1883.xlsx'
#sheet_name='T30.012'

## Function Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy.ma as ma
from numpy import diff
from scipy.ndimage import interpolation
import math
from sklearn import linear_model
import statsmodels.api as sm
from scipy.optimize import curve_fit


### Data Mining- Calculator Function
Obtains all caclulations for whichever loop (or loops if using a "for loop" to run through multiple) are used. You only need to add parameters to this block (and update the outputs from when you call this)if you want to mine for more data.

In [ ]:
def Calculator(time_step,Loop,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData):
    #Calculate the various physiological parameters
    #Obtain the stroke work
    
    SW=PolyArea(Loop[0],Loop[1])

    #Obtain the stroke volume
    SV=EndDiastoleData[0]-EndSystoleData[0]

    #Obtain the ejection fraction
    EJF=SV/int(EndDiastoleData[0])*100
    
    #Obtain the cardiac output
    HR=60/(len(Loop[0])*time_step)
    CO=SV*HR
    
    #Obtain the elastance
    Ea=EndSystoleData[1]/SV
    
    #Max and Min Pressures
    Pmax=np.max(Loop[1])
    Pmin=np.min(Loop[1])

    #Max and Min Volumes
    Vmax=np.max(Loop[0])
    Vmin=np.min(Loop[0])

    return SW,SV,EJF,HR,CO,Ea, Pmax, Pmin, Vmax, Vmin

### Data Processing Functions

In [ ]:
#Define cartesian to polar conversion.
def cart2pol(x, y):
    theta = np.arctan2(y, x)
    radius = np.sqrt(x**2 + y**2)
    return(theta, radius)

#Define polar to cartesian conversion.
def pol2cart(theta, radius):
    x = radius * np.cos(theta)
    y = radius * np.sin(theta)
    return(x, y)

#Define Shoelace Formula
def PolyArea(x_array,y_array):
    return 0.5*np.abs(np.dot(x_array,np.roll(y_array,1))-np.dot(y_array,np.roll(x_array,1)))

#Estimate the centroid locations of the data points in cartesian coordinates. 
def centroid_calculator(loop):
#    print('x_coordinates are')
#    print(loop[0])
#    print('y_coordinates are')
#    print(loop[1])
    x_centroid=np.mean(loop[0])
    y_centroid=np.mean(loop[1])
    centroid=[x_centroid, y_centroid]
    return centroid

#Create a backup centroid calculator in cases where there is skewed data
def centroid_calculator2(loop):
#    print('x_coordinates are')
#    print(loop[0])
#    print('y_coordinates are')
#    print(loop[1])
    x_centroid=(np.max(loop[0])+np.min(loop[0]))/2
    y_centroid=(np.max(loop[1])+np.min(loop[1]))/2
    centroid=[x_centroid, y_centroid]
    return centroid

#Define a function to mask the loops based on what quadrant is desired
def PV_Loop_Mask(loop,mask_value,unmasked_direction):
  #make sure first column is x values and second column is y values
    x=loop[:,0]
    y=loop[:,1]
    indices=loop[:,2]
    loop_indices=[]
    loop_x=[]
    loop_y=[]

    if unmasked_direction=='+x':
        loop=ma.masked_array(x,mask=((x<mask_value)))
    elif unmasked_direction=='-x':
        loop=ma.masked_array(x,mask=((x>mask_value)))
    elif unmasked_direction=='+y':
        loop=ma.masked_array(y,mask=((y<mask_value)))
    elif unmasked_direction=='-y':
        loop=ma.masked_array(y,mask=((y>mask_value)))    

    for i in range(len(loop)):
        if loop[i]!='--':
            loop_indices.append(indices[i])
            loop_x.append(x[i])
            loop_y.append(y[i])
    loop_indices=np.array(loop_indices)
    loop_x=np.array(loop_x)
    loop_y=np.array(loop_y)
    masked_loop=np.column_stack((loop_x,loop_y,loop_indices))
    return masked_loop


#Code to obtain the derivative of each individual sub-loop to find values such as dP/dt (and dV/dt, if needed)
def d_dt(loop):
    derivatives=[]
    x=loop[:,0]
    y=loop[:,1]
    np.seterr(invalid='ignore')
    t=np.cumsum(np.diff(x)**2+np.diff(y)**2)
    t=np.insert(t,0,0)
    t=np.transpose(t)

    dxdt=diff(x)/diff(t)
    dydt=diff(y)/diff(t)
    
    derivatives=[dxdt,dydt,loop[:,2]]

    return derivatives

def HistogramGenerator(data,name):

    fig = plt.figure()
    fig.patch.set_facecolor('pink')
    fig.patch.set_alpha(0.3)
    ax = fig.add_subplot(111)
    
    n, bins, patches = plt.hist(x=data, bins='auto',alpha=0.6, rwidth=0.85)
    plt.grid(axis='y', alpha=0.65)
    plt.xlabel('{}'.format(name), color='red')
    plt.ylabel('Frequency', color='blue')
    plt.title('{} Frequency'.format(name), color='green')
    print('Mean {}:'.format(name))
    print(np.mean(data))

def LinePlotter(data,name):

    fig = plt.figure()
    fig.patch.set_facecolor('purple')
    fig.patch.set_alpha(0.15)
    ax = fig.add_subplot(111)
    plt.grid(axis='y', alpha=0.65)
    plt.xlabel('Loop Number')
    plt.ylabel('{} Values'.format(name))
    #plt.xticks(np.arange(min(x), max(x)+1, 1.0))
    plt.title('{} Values as a function of PV Loop'.format(name), color='green')
    plt.plot(data,linestyle=':', marker='*', color='blue')
    plt.show()  
    
def LinePlotter2(x_axis,data,name):

    fig = plt.figure()
    fig.patch.set_facecolor('green')
    fig.patch.set_alpha(0.2)
    ax = fig.add_subplot(111)
    plt.grid(axis='y', alpha=0.65)
    plt.xlabel('Time Point (seconds)')
    plt.ylabel('{} Values'.format(name))
    plt.title('{} Values as a function of Time (in seconds)'.format(name), color='green')
    plt.plot(x_axis,data,linestyle=':', marker='*', color='green')
    plt.show()
    print('This data spans over {} seconds'.format(x_axis[-1]))

def RawHemodynamicPlotter (x,x_name,y,y_name,color,intensity):
    fig = plt.figure()
    fig.patch.set_facecolor(color)
    fig.patch.set_alpha(intensity)
    ax = fig.add_subplot(111)
    plt.plot(x,y)
    plt.title('{} vs {}'.format (y_name,x_name))
    plt.xlabel('{}'.format(x_name))
    plt.ylabel('{}'.format(y_name))
    plt.show()

def ResizeArray (np_array, desired_size):
    z = desired_size / len(np_array)
    reshaped_np_array = interpolation.zoom(np_array,z)
    return reshaped_np_array

def PVLoopRecenter(PV_Loop_Start, volume,pressure):
    ShadowLoop_start=[]
    
    #Catching "shadow" PV loops
    for step in range(len(PV_Loop_Start)-1):
        
        print('You are in step number: {}'.format(step))

        indexed_start=PV_Loop_Start[step]
        #Start of new loop
        a=int(PV_Loop_Start[step]-10)
        #Stop of new loop
        b=int(PV_Loop_Start[step+1]+1)
        
        #Temporary
        print('This fake loop starts at {} and stops at {}'.format(a,b))
        
        #Create dummy volume and pressures in cartesian coordinates
        dummy_volume=volume[a:b]
        dummy_pressure=pressure[a:b]
#        np.append(volume,vol[a+])
#        np.append(pressure,press[a+2])
        #Combine them into a single dummy_loop
        dummy_loop_cartesian=[dummy_volume,dummy_pressure]
        dummy_centroid=centroid_calculator(dummy_loop_cartesian)
        #Convert the dummy loop components into polar coordinates again
        dummy_theta_array,dummy_radius_array=cart2pol(dummy_volume-dummy_centroid[0],dummy_pressure-dummy_centroid[1])
        
    
        #Create empty lists for instances where there are:
        
        #Crosses for Quadrant 4 to Quadrant 1
        QuadIV_I=[]
        #Crosses for Quadrant 1 to Quadrant 2
        QuadI_II=[]
        #Crosses for Quadrant 2 to Quadrant 3
        QuadII_III=[]
        #Crosses for Quadrant 3 to Quadrant 4
        QuadIII_IV=[]
        
        #This is where we actually populate the lists with data:
        for step2 in range(1, len(dummy_radius_array)):
            #Cross Quad 4 to 1
            if dummy_theta_array[step2]>=0 and dummy_theta_array[step2-1]<0:
                QuadIV_I.append(step2+indexed_start)
            #Cross Quad 1 to 2
            if dummy_theta_array[step2]>=math.pi/2 and dummy_theta_array[step2-1]<math.pi/2:
                QuadI_II.append(step2+indexed_start)        
            #Cross Quad 2 to 3
            if dummy_theta_array[step2]<=0 and dummy_theta_array[step2-1]>0:
                QuadII_III.append(step2+indexed_start)
            #Cross Quad 3 to 4
            if dummy_theta_array[step2]>=0 and dummy_theta_array[step2-1]<0:
                QuadIII_IV.append(step2+indexed_start)
        print('Preprocesed QuadIV_I value= {}'.format(QuadIV_I))
        #print(QuadI_II)
        #print(QuadII_III)
        #print(QuadIIII_IV)
        chosen_start=np.min(QuadIV_I)
        ShadowLoop_start.append(chosen_start)
        print('We have added {} to the shadow loop'.format(np.min(QuadIV_I)))
        #This section exists so that we can sift through our quadrant crossing and seperate loops:
        
        
        #Graphing for testing purposes
#        fig = plt.figure()
#        fig.patch.set_facecolor('pink')
#        fig.patch.set_alpha(0.9)
#        ax = fig.add_subplot(111)
#        plt.plot(dummy_volume,dummy_pressure)
#        plt.plot(dummy_centroid[0],dummy_centroid[1], linestyle=':', marker='*', color='red')
#        plt.title('Pseudo Loop Number {}'.format(step))
#        plt.plot(dummy_volume[chosen_start-indexed_start],dummy_pressure[chosen_start-indexed_start], linestyle=':', marker='*', color='red')        
#        plt.show()

    #PV_Loop_Start=[]
    PV_Loop_Start=ShadowLoop_start
    #return PV_Loop_Start
    return PV_Loop_Start

def hunt_bad_loops(Physiological_Loops,Cycle_Start,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt,Loop_Size,correction_factor):
    #correction_factor=1
    bad_loop_hunter = input('Are there bad PV loops? Type "Yes" or "No." ')

    while bad_loop_hunter!='No' or bad_loop_hunter!='NO' or bad_loop_hunter!='no' or bad_loop_hunter!='nO':
        bad_loop_hunter = input('Enter their corresponding loop number(s). Type "No" if there are none.  ')
        if bad_loop_hunter=='No' or bad_loop_hunter=='NO' or bad_loop_hunter=='no' or bad_loop_hunter=='nO':
            break
        bad_loop_hunter=int(bad_loop_hunter)-correction_factor
        Cycle_Start.pop(bad_loop_hunter)
        Physiological_Loops.pop(bad_loop_hunter)
        Loop_Size.pop(bad_loop_hunter)
        EndSystoleData.pop(bad_loop_hunter)
        EndDiastoleData.pop(bad_loop_hunter)
        EndSystole_ddt.pop(bad_loop_hunter)
        EndDiastole_ddt.pop(bad_loop_hunter)
        correction_factor=correction_factor+1
        print('This number has been removed')
        #bad_loop_hunter = input('Enter their corresponding loop number(s). Type "No" if there are none remaining.')
    print('All bad loops have been removed.')
    print('Look here: ')
    
    
    for step in range(len(Physiological_Loops)):
        fig = plt.figure()
        fig.patch.set_facecolor('green')
        fig.patch.set_alpha(0.1)
        ax = fig.add_subplot(111)
        # Isolate an individual loop
        individual_loop=Physiological_Loops[step]
        # Convert the polar coordinates into x (volume) and y (pressure)
        dummy_centroid=centroid_calculator(Physiological_Loops)
        #plt.plot(Physiological_Loops[step][0], Physiological_Loops[step][1], linestyle=':', color='green')
        plt.plot(Physiological_Loops[step][0], Physiological_Loops[step][1], color='green')
        plt.xlabel('Volume')
        plt.ylabel('Pressure')
        plt.title('Physiological PV Loop {}'.format(step+1))
        plt.plot(dummy_centroid[0],dummy_centroid[1],color='red')
        plt.show()
    
    return Physiological_Loops,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt

def check_nth_column_legality(n,loop):
    pass_fail=0
    column_search=loop[n]
    for i in range(len(column_search)):
        if column_search[i] != 'nan' or columnn_search[i] != 0:
            pass_fail=1
            break
    return pass_fail

def corner_hunter(loop,centroid):
    dummy_loop=[loop[:,0],loop[:,1]]
    distances=[]
    counter=[]
    for i in range(len(loop)):
        x_dist=(loop[i,0]-centroid[0])**2
        y_dist=(loop[i,1]-centroid[1])**2
        dummy_distance=(x_dist+y_dist)**.5
        distances.append(dummy_distance)
        counter.append(loop[i,2])
    distances=[distances, counter]
    return distances

## Actual Code

### Import Raw Data and Plot It

In [ ]:
#Import Raw Data
PV_Raw = pd.read_excel (file_name, sheet_name)
#PV_Raw = pd.read_excel(file_name)
#Convert Raw Data into a numpy array

PV_Raw=PV_Raw.to_numpy()


#Obtain the Volume and Pressure Data
vol=PV_Raw[:,volume_index]
press=PV_Raw[:,pressure_index]



#Making a  time array
time_array=np.linspace(0,(int(len(vol))+1)*time_step,int(len(vol)))


RawHemodynamicPlotter(vol,'Volume',press,'Pressure','pink',0.2)
RawHemodynamicPlotter(time_array,'Time',vol,'Volume','blue',.1)
RawHemodynamicPlotter(time_array,'Time',press,'Pressure','green',.1)


### Manipulate the data to get individual "pseudo-loops"

In [ ]:
#Define the origin from the averages of all of the points
Pressure_origin=np.mean(press)
Volume_origin=np.mean(vol)

#Convert from cartesian to polar coordinates
theta_array,radius_array=cart2pol(vol-Volume_origin,press-Pressure_origin)

#Determine where there is a "new" PV loop based on an arbitrary x-axis relative to the "origin"
PV_Loop_Start=[]
#Define a 'for' loop over the entire length of the array to determine where the "new" PV loops start
for step in range(1, len(radius_array)):
    if theta_array[step]>=0 and theta_array[step-1]<0:
        #Just as seen in Stonko's code, we will use an arbitrary x-axis where a shift in the signs for the theta from positive to negative values
        PV_Loop_Start.append(step)
print('The PV Loops Start at the following places: {}'.format(PV_Loop_Start))



## If you find low fidelity loops, run this recentering algorithm

#Use the PVLoopRecenter algorithm to account for errors
#<br>
PV_Loop_Start=PVLoopRecenter(PV_Loop_Start, vol,press)

In [ ]:
#Seperate Loops into Individual Lists Elements
Loops=[]
for step in range(len(PV_Loop_Start)-1):
    #Start of new loop
    a=PV_Loop_Start[step]
    #Stop of new loop
    #b=PV_Loop_Start[step+1]+1
    b=PV_Loop_Start[step+1]
    dummy_vol=np.array(vol[a:b])
    dummy_press=np.array(press[a:b])
    dummy_loop=np.column_stack((dummy_vol,dummy_press))
    Loops.append(dummy_loop)



####  code to see if there are abnormally small PV loops 

for a in range(1,len(PV_Loop_Start)):

    print('Loop #{}'.format(a))
    
    print(PV_Loop_Start[a]-PV_Loop_Start[a-1])

### Calculate all of the End Systolic and End Diastolic Information by re-evaluating pseudo-loops

In [ ]:
EndSystoleData=[]
EndDiastoleData=[]
EndSystole_ddt=[]
EndDiastole_ddt=[]


zoom_factor=5


for step in range(len(PV_Loop_Start)-1):
#for step in range(len(PV_Loop_Start)):
    #Set up the plot
    fig = plt.figure()
    fig.patch.set_facecolor('pink')
    fig.patch.set_alpha(0.9)
    ax = fig.add_subplot(111)
    ax.set_xlabel('Volume')
    ax.set_title('Fake PV Plot {}'.format(step+1))
    ax.set_ylabel('Pressure')

    
    # Create the dummy loop
    x=np.array(Loops[step][:,0])
    y=np.array(Loops[step][:,1])
    plt.plot(x,y,color='green')
    dummy_loop=[x,y]
    
    #Get the centroid
    centroid=centroid_calculator(dummy_loop)
    plt.plot(centroid[0],centroid[1], color='purple',marker='*')
    #Create an array with all of the individual loop element values *relative to each individual loop*
    loop_element=[]
    for i in range(len(x)):
        loop_element.append(i)

    # Recreate the dummy loop to include the loop element
    dummy_loop=np.column_stack((x,y,loop_element))

    
    #Define the upper loop
    upper_loop=PV_Loop_Mask(dummy_loop,centroid[1],'+y')
    #Then isolate the upper left portion of the loop
    upper_left_loop=PV_Loop_Mask(upper_loop,centroid[0],'-x')

    #Define the lower loop
    lower_loop=PV_Loop_Mask(dummy_loop,centroid[1],'-y')
    #Then isolate the lower right portion of the loop
    lower_right_loop=PV_Loop_Mask(lower_loop,centroid[0],'+x')
    
    #Plot unprocessed upper left and lower right loops
    plt.plot(upper_left_loop[:,0],upper_left_loop[:,1],color='red')
    plt.plot(lower_right_loop[:,0],lower_right_loop[:,1],color='red')
    
    #To see if the dP/dt algorithm will work by checking the second (n=1) column
    upper_left_derivatives=d_dt(upper_left_loop)
    upper_left_pass_fail=check_nth_column_legality(1,upper_left_derivatives)
    upper_left_centroid=centroid
    
    lower_right_derivatives=d_dt(lower_right_loop)
    lower_right_pass_fail=check_nth_column_legality(1,lower_right_derivatives)
    lower_right_centroid=centroid

    #Begin Zooming 
    
    
    ###############################################################################################    
    #Zoom the upper left side
    i=0

    while len(upper_left_loop)>len(dummy_loop)/zoom_factor:
        dummy_upper_left_loop=[upper_left_loop[:,0],upper_left_loop[:,1]]
        centroid2=centroid_calculator(dummy_upper_left_loop)
        dummy_upper_left_loop=upper_left_loop
        dummy_upper_left_loop=PV_Loop_Mask(dummy_upper_left_loop,centroid2[0],'-x')
        dummy_upper_left_loop=PV_Loop_Mask(dummy_upper_left_loop,centroid2[1],'+y')
        if len(dummy_upper_left_loop)>1:
            size_prev=len(upper_left_loop)
            size_new=len(dummy_upper_left_loop)
            if size_prev==size_new:
                break
            else:
                upper_left_loop=dummy_upper_left_loop
                upper_left_centroid=centroid2
                i=i+1
        else:
            break
    plt.plot(upper_left_centroid[0], upper_left_centroid[1], linestyle=':', marker='x', color='aqua')
    print('The upper left loop was zoomed {} times'.format(i)) 
    plt.plot(upper_left_loop[:,0], upper_left_loop[:,1], linestyle=':', marker='.', color='blue') 
    
    if upper_left_pass_fail==1:
        print('Angel')
        derivatives=d_dt(upper_left_loop)

        #Note- making an assumption it is abs value
        #index=np.argmax(abs(derivatives[1]))
        index=np.argmin(abs(derivatives[1]))
        #Save this data to the End Systole_ddt array
        ESddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
        EndSystole_ddt.append(ESddt)


        index=derivatives[2][index]
        index=int(index)

        hold=dummy_loop[index,:] 

        hold=hold.tolist()
        EndSystoleData.append(hold) 
    else:
        print('Devil')
        top_right_distances=corner_hunter(upper_left_loop,upper_left_centroid)
        max_dist=np.max(top_right_distances[0])
        point_max_dist=np.argmax(top_right_distances[0])
        index=int(top_right_distances[1][point_max_dist])

        hold=dummy_loop[index,:]
        #Save this data to the End Systole_ddt array
        ESddt=[0,0,hold]
        EndSystole_ddt.append(ESddt)
        hold=dummy_loop[int(hold[2]),:] 
        hold=hold.tolist()
        EndSystoleData.append(hold) 
    plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')
    
    ###############################################################################################

    

    
    
    ###############################################################################################    
    #Zoom the lower right side
    i=0

    while len(lower_right_loop)>len(dummy_loop)/zoom_factor:
        dummy_lower_right_loop=[lower_right_loop[:,0],lower_right_loop[:,1]]
        centroid2=centroid_calculator(dummy_lower_right_loop)
    
        dummy_lower_right_loop=lower_right_loop
        dummy_lower_right_loop=PV_Loop_Mask(dummy_lower_right_loop,centroid2[0],'+x')
        dummy_lower_right_loop=PV_Loop_Mask(dummy_lower_right_loop,centroid2[1],'-y')
        if len(dummy_lower_right_loop)>1:
            size_prev=len(lower_right_loop)
            size_new=len(dummy_lower_right_loop)
            if size_prev==size_new:
                break
            else:
                lower_right_loop=dummy_lower_right_loop
                i=i+1
                lower_right_centroid=centroid2
                
        else:    
            break
    plt.plot(lower_right_centroid[0], lower_right_centroid[1], linestyle=':', marker='x', color='aqua')
    print('The lower right loop was zoomed {} times'.format(i)) 
    plt.plot(lower_right_loop[:,0], lower_right_loop[:,1], linestyle=':', marker='.', color='blue')

    
    if lower_right_pass_fail==1:
        print('Angel')
        derivatives=d_dt(lower_right_loop)

        #Note- making an assumption it is abs value
        #index=np.argmax(abs(derivatives[1]))
        index=np.argmax((derivatives[1]))
        #Save this data to the End Systole_ddt array
        EDddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
        EndDiastole_ddt.append(ESddt)


        index=derivatives[2][index]
        index=int(index)

        hold=dummy_loop[index,:] 
        hold=hold.tolist()
        EndDiastoleData.append(hold) 
    else:
        print('Devil')
        bottom_right_distances=corner_hunter(lower_right_loop,centroid)
        #print(bottom_right_distances)
        #check_bottom_right=check_nth_column_legality(bottom_right_distances,0)
        #if check_bottom_right==1:
        if bool(bottom_right_distances[0]):
            max_dist=np.max(bottom_right_distances[0])
            point_max_dist=np.argmax(bottom_right_distances[0])
            index=int(bottom_right_distances[1][point_max_dist])
            
            
        else:
            print('Super Devil')
            total_distances=corner_hunter(dummy_loop,centroid)
            max_dist=np.max(total_distances[0])
            point_max_dist=np.argmax(total_distances[0])
            index=int(total_distances[1][point_max_dist])
            
            
            
            
            
        hold=dummy_loop[index,:]
        #Save this data to the End Diatole_ddt array
        EDddt=[0,0,hold]
        EndDiastole_ddt.append(ESddt)

        hold=dummy_loop[int(hold[2]),:] 
        hold=hold.tolist()
        EndDiastoleData.append(hold) 
    plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')
    
    ###############################################################################################
 
    
    plt.show()


### Process End Systolic and End Diastolic data to get 'real' PV Loops

In [ ]:
PV_Loop_Start.pop(0)
for i in range(len(EndSystoleData)-1):
    EndSystoleData[i+1][2]=EndSystoleData[i+1][2]+(PV_Loop_Start[i])
    EndDiastoleData[i+1][2]=EndDiastoleData[i+1][2]+(PV_Loop_Start[i])

#EndSystoleData.pop(0)
#EndDiastoleData.pop(0)
#EndSystole_ddt.pop(0)
#EndDiastole_ddt.pop(0)


Cycle_Start=[]
Physiological_Loops=[]
Loop_Size=[]
dummy_HR=[]
for step in range(len(EndSystoleData)-1):
#for step in range(len(EndSystoleData)):

#The first loop is often never complete,so skipping it
    if step==len(EndSystoleData)-1:

        continue
    #Start of new loop
    a=int(EndSystoleData[step][2])
    a=a-1 
    
    #Stop of new loop
    b=int(EndSystoleData[step+1][2])
    b=b+1
    
    #Save the start times
    Cycle_Start.append(EndSystoleData[step][2]*time_step)
    
    #Obtain the Volume and Pressure Data
    volume=vol[a:b]
    volume=np.append(volume,vol[a+1])
    pressure=press[a:b]
    pressure=np.append(pressure,press[a+1])

    
    Loop_Size.append(len(volume))
    
    fig = plt.figure()
    fig.patch.set_facecolor('yellow')
    fig.patch.set_alpha(0.1)
    ax = fig.add_subplot(111)
    #Plot it
    plt.plot(volume, pressure,marker= '*')
    
    #Create a dummy loop to add to the list of the Physiological PV loops
    dummy_loop=[volume,pressure]
    Physiological_Loops.append(dummy_loop)

    #For Visualization purposes, show the centroid
    centroid=centroid_calculator(dummy_loop)
    plt.plot(centroid[0], centroid[1], linestyle=':', marker='*', color='red')
    plt.title('Candidate Physiological PV Loop {}'.format(step))
    
    plt.plot(EndSystoleData[step][0], EndSystoleData[step][1], linestyle=':', marker='*', color='red')
    plt.plot(EndDiastoleData[step][0], EndDiastoleData[step][1], linestyle=':', marker='*', color='red')
    
    #Show all plots
    plt.show()
    
    
    #User selection guide
    print('P-V Number of points: {}'.format(b-a+1))
    print('The Heart Rate is: {} BPM'.format(60/((b-a+1)*time_step)))
    dummy_HR.append(60/((b-a+1)*time_step))
    print('The ESV is {} and the ESP is {} at point {}.'.format(EndSystoleData[step][0], EndSystoleData[step][1], EndSystoleData[step][2]))
    if EndSystoleData[step][0]<0:
        print('Negative systolic volume detected')
        
    if EndSystoleData[step][1]<0:
        print('Negative systolic pressure detected')
        
    print('The EDV is {} and the EDP is {} at point {}.'.format(EndDiastoleData[step][0], EndDiastoleData[step][1],EndDiastoleData[step][2]))
    if EndDiastoleData[step-1][0]<0:
        print('Negative diastolic volume detected')
        
    if EndDiastoleData[step-1][1]<0:
        print('Negative diastolic pressure detected')

In [ ]:
print(Loop_Size)
HistogramGenerator(Loop_Size,'Loop Sizes')

In [ ]:
print(dummy_HR)
HistogramGenerator(dummy_HR,'Heart Rates')

## Manually Remove the Garbage PV Loops

In [ ]:
#if sheet_name=='T0.1' or sheet_name=='T30.1' or sheet_name=='T74.1':
Physiological_Loops,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt=hunt_bad_loops(Physiological_Loops,Cycle_Start,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt,Loop_Size,0)



Physiological_Loops,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt=hunt_bad_loops

(Physiological_Loops,Cycle_Start,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt,Loop_Size,1)



print('Round 2 of bad loop hunt')
ask=input('Are there any more bad loops? Type "Yes" if there are. ')
if ask=='Yes' or ask=='yes':
    Physiological_Loops,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt=hunt_bad_loops(Physiological_Loops,Cycle_Start,EndSystoleData,EndDiastoleData,Loop_Size,0)
    

In [ ]:
# Mine the End Systolic and Diastolic Data

#Remove the first and last points

#The first is ignored because it is usually incomplete
#EndSystoleData.pop(0)
#EndDiastoleData.pop(0)

#The last is removed because it is an end point, and usually followed by an also incomplete loop
EndSystoleData.pop(-1)
EndDiastoleData.pop(-1)

ESV_total=[]
ESP_total=[]
EDV_total=[]
EDP_total=[]
dPmax=[]
dPmin=[]



for step in range(len(EndSystoleData)):
    
    ESV_total.append(EndSystoleData[step][0])
    ESP_total.append(EndSystoleData[step][1])

    EDV_total.append(EndDiastoleData[step][0])
    EDP_total.append(EndDiastoleData[step][1])
    
    dPmax.append(EndSystole_ddt[step][1])
    dPmin.append(EndDiastole_ddt[step][1])


# Obtain Hemodynamic Information

In [ ]:
#Heart Rate pre-Calculation
#Number of loops=number of heart beats
number_of_loops=len(EndSystoleData)
#Total time measured (in minutes)
total_time=time_array[-1]/60

## Dynamic Function that calculates Hemodynamic Parameters

In [ ]:
#Seperate Loops into Individual Lists Elements

StrokeWork=[]
StrokeVolume=[]
EjectionFraction=[]
Heart_Rate=[]
CardiacOutput=[]
Elastance=[]
MaxPressure=[]
MinPressure=[]
MaxVolume=[]
MinVolume=[]


for step in range(len(Physiological_Loops)):
    SW,SV,EJF,HR,CO,Ea, Pmax, Pmin, Vmax, Vmin=Calculator(time_step,Physiological_Loops[step],Cycle_Start,Loop_Size[step],EndSystoleData[step],EndDiastoleData[step])
    StrokeWork.append(SW)
    StrokeVolume.append(SV)
    EjectionFraction.append(EJF)
    Heart_Rate.append(HR)
    CardiacOutput.append(CO)
    Elastance.append(Ea)
    MaxPressure.append(Pmax)
    MinPressure.append(Pmin)
    MaxVolume.append(Vmax)
    MinVolume.append(Vmin)

In [ ]:
HistogramGenerator(StrokeWork,'Stroke Work (mL*mmHg)')
LinePlotter(StrokeWork,'Stroke Work  (mL*mmHg)')
LinePlotter2(Cycle_Start,StrokeWork,'Stroke Work  (mL*mmHg)')

 

HistogramGenerator(StrokeVolume,'Stroke Volume (mL)')
LinePlotter(StrokeVolume,'Stroke Volume (mL)')
LinePlotter2(Cycle_Start,StrokeVolume,'Stroke Volume (mL)')

 


HistogramGenerator(Elastance,'Elastance')
LinePlotter(Elastance,'Elastance')
LinePlotter2(Cycle_Start,Elastance,'Elastance')

 


HistogramGenerator(CardiacOutput,'Cardiac Output')
LinePlotter(CardiacOutput,'Cardiac Output')
LinePlotter2(Cycle_Start,CardiacOutput,'Cardiac Output')

 

HistogramGenerator(EjectionFraction,'Ejection Fraction')
LinePlotter(EjectionFraction,'Ejection Fraction')
LinePlotter2(Cycle_Start,EjectionFraction,'Ejection Fraction')


HistogramGenerator(Heart_Rate,'Heart Rates')
LinePlotter(Heart_Rate,'Heart Rates')
LinePlotter2(Cycle_Start,Heart_Rate,'Heart Rates')

HistogramGenerator(MaxPressure,'Max Pressure (mmHg)')
LinePlotter(MaxPressure,'Max Pressure (mmHg)')
LinePlotter2(Cycle_Start,MaxPressure,'Max Pressure (mmHg)')

 

HistogramGenerator(MinPressure,'Min Pressure (mmHg)')
LinePlotter(MinPressure,'Min Pressure (mmHg)')
LinePlotter2(Cycle_Start,MinPressure,'Min Pressure (mmHg)')

 

HistogramGenerator(MaxVolume,'Max Volume (mL)')
LinePlotter(MaxVolume,'Max Volume (mL)')
LinePlotter2(Cycle_Start,MaxVolume,'Max Volume (mL)')

 

HistogramGenerator(MinVolume,'Min Volume (mL)')
LinePlotter(MinVolume,'Min Volume (mL)')
LinePlotter2(Cycle_Start,MinVolume,'Min Volume (mL)')

 

HistogramGenerator(ESV_total,'ESV (mL)')
LinePlotter(ESV_total,'ESV (mL)')
LinePlotter2(Cycle_Start,ESV_total,'ESV (mL)')

 

HistogramGenerator(ESP_total,'ESP (mmHg)')
LinePlotter(ESP_total,'ESP (mmHg)')
LinePlotter2(Cycle_Start,ESP_total,'ESP (mmHg)')

 

HistogramGenerator(EDV_total,'EDV (mL)')
LinePlotter(EDV_total,'EDV (mL)')
LinePlotter2(Cycle_Start,EDV_total,'EDV (mL)')

 

HistogramGenerator(EDP_total,'EDP (mmHg)')
LinePlotter(EDP_total,'EDP (mmHg)')
LinePlotter2(Cycle_Start,EDP_total,'EDP (mmHg)')

In [ ]:
print('Raw Mean Data Information')

rawmean_SW=np.mean(StrokeWork)
print('The Raw Mean of Stroke Work is {}'.format(rawmean_SW))

rawmean_SV=np.mean(StrokeVolume)
print('The Raw Mean of the Stroke Volume is {}'.format(rawmean_SV))

rawmean_EJF=np.mean(EjectionFraction)
print('The Raw Mean of the Ejection Fraction is {}'.format(rawmean_EJF))

rawmean_HR=np.mean(Heart_Rate)
print('The Raw Mean of the Heart Rate is {}'.format(rawmean_HR))

rawmean_CO=np.mean(CardiacOutput)
print('The Raw Mean of the Cardiac Output is {}'.format(rawmean_CO))

rawmean_Elastance=np.mean(Elastance)
print('The Raw Mean of the Elastance is {}'.format(rawmean_Elastance))

rawmean_MaxPressure=np.mean(list(MaxPressure))
print('The Raw Mean of the Max Pressure is {}'.format(rawmean_MaxPressure))

rawmean_MinPressure=np.mean(list(MinPressure))
print('The Raw Mean of the Min Pressure is {}'.format(rawmean_MinPressure))

rawmean_MaxVolume=np.mean(list(MaxVolume))
print('The Raw Mean of the Max Volume is {}'.format(rawmean_MaxVolume))

rawmean_MinVolume=np.mean(list(MinVolume))
print('The Raw Mean of the Min Volume is {}'.format(rawmean_MinVolume))

rawmean_ESV=np.mean(list(ESV_total))
print('The Raw Mean of the ESV is {}'.format(rawmean_ESV))

rawmean_ESP=np.mean(list(ESP_total))
print('The Raw Mean of the ESP is {}'.format(rawmean_ESP))

rawmean_EDV=np.mean(list(EDV_total))
print('The Raw Mean of the EDV is {}'.format(rawmean_EDV))

rawmean_EDP=np.mean(list(EDP_total))
print('The Raw Mean of the EDP is {}'.format(rawmean_EDP))

# Ensemble Average

In [ ]:
#Code to obtain the ensemble average

avg_loop_size=np.mean(Loop_Size)
avg_loop_size=np.rint(avg_loop_size)

max_vol=[]
min_vol=[]
max_press=[]
min_press=[]

ensemble_volume=np.zeros(shape=(int(avg_loop_size)-1,len(Physiological_Loops)))
ensemble_pressure=np.zeros(shape=(int(avg_loop_size)-1,len(Physiological_Loops)))



for step in range(len(Physiological_Loops)):
    x=np.array(Physiological_Loops[step])
    dummy_volume=ResizeArray (x[0], avg_loop_size-1)
    ensemble_volume[:,step]=dummy_volume
    dummy_pressure=ResizeArray (x[1], avg_loop_size-1)
    ensemble_pressure[:,step]=dummy_pressure

    

for step in range(len(ensemble_volume)):
    max_vol.append(np.max(ensemble_volume[step]))
    min_vol.append(np.min(ensemble_volume[step]))
    max_press.append(np.max(ensemble_pressure[step]))
    min_press.append(np.min(ensemble_pressure[step]))
    
    
#Determine the ensemble average from all of this data

ensemble_volume=np.mean(ensemble_volume, axis=1)
ensemble_pressure=np.mean(ensemble_pressure, axis=1)

#Close the loop
ensemble_volume=np.append(ensemble_volume,ensemble_volume[0])
ensemble_pressure=np.append(ensemble_pressure,ensemble_pressure[0])




#Resize
ensemble_volume=ResizeArray (ensemble_volume, avg_loop_size)
ensemble_pressure=ResizeArray (ensemble_pressure, avg_loop_size)

for i in range(len(ensemble_volume)):
    if ensemble_volume[i]==0 and ensemble_pressure[i]==0:
        ensemble_volume=np.delete(ensemble_volume,i)
        ensemble_pressure=np.delete(ensemble_pressure,i)


ensemble_loop=[ensemble_volume,ensemble_pressure]

for i in range(len(ensemble_loop)):
    if ensemble_loop[i][0]==0 and ensemble_loop[i][1]==0:
        ensemble_loop.pop(i)

ESPV_ensemble=[]
EDPV_ensemble=[]

fig = plt.figure()
fig.patch.set_facecolor('blue')
fig.patch.set_alpha(0.1)
ax = fig.add_subplot(111)

#Plot the ensemble average
plt.plot(ensemble_volume, ensemble_pressure, linestyle=':', marker='*', color='green')

#plt.fill_between(ensemble_pressure, min_vol, max_vol)

#plt.plot(ensemble_volume, ensemble_pressure, color='green')
plt.xlabel('Volume')
plt.ylabel('Pressure')
plt.title('Ensemble Averaged Pressure vs Volume with {} points'.format(int(avg_loop_size)))
plt.show()


## Obtain all the Ensemble Ouptuts

##### Ensemble ESPV and EDPV collection

In [ ]:
fig = plt.figure()
fig.patch.set_facecolor('blue')
fig.patch.set_alpha(0.1)
ax = fig.add_subplot(111)


ESPV_ensemble=[]
EDPV_ensemble=[]

zoom_factor=8
#Create an array with all of the individual loop element values *relative to each individual loop*

centroid=centroid_calculator(ensemble_loop)

loop_element=[]
for i in range(len(ensemble_volume)):
    loop_element.append(i)
# Recreate the dummy loop to include the loop element
dummy_loop=np.column_stack((ensemble_loop[0],ensemble_loop[1],loop_element))




# Create the dummy loop
x=np.array(ensemble_loop[0])
y=np.array(ensemble_loop[1])
plt.plot(x,y,color='green')
dummy_loop=[x,y]

#Get the centroid
centroid=centroid_calculator(dummy_loop)
plt.plot(centroid[0],centroid[1], color='purple',marker='*')
#Create an array with all of the individual loop element values *relative to each individual loop*
loop_element=[]
for i in range(len(x)):
    loop_element.append(i)

# Recreate the dummy loop to include the loop element
dummy_loop=np.column_stack((x,y,loop_element))


#Define the upper loop
upper_loop=PV_Loop_Mask(dummy_loop,centroid[1],'+y')
#Then isolate the upper left portion of the loop
upper_left_loop=PV_Loop_Mask(upper_loop,centroid[0],'-x')

#Define the lower loop
lower_loop=PV_Loop_Mask(dummy_loop,centroid[1],'-y')
#Then isolate the lower right portion of the loop
lower_right_loop=PV_Loop_Mask(lower_loop,centroid[0],'+x')

#Plot unprocessed upper left and lower right loops
plt.plot(upper_left_loop[:,0],upper_left_loop[:,1],color='red')
plt.plot(lower_right_loop[:,0],lower_right_loop[:,1],color='red')

#To see if the dP/dt algorithm will work by checking the second (n=1) column
upper_left_derivatives=d_dt(upper_left_loop)
upper_left_pass_fail=check_nth_column_legality(1,upper_left_derivatives)

lower_right_derivatives=d_dt(lower_right_loop)
lower_right_pass_fail=check_nth_column_legality(1,lower_right_derivatives)

#Begin Zooming
zoom_factor=len(dummy_loop)/zoom_factor

###############################################################################################    
#Zoom the upper left side
i=0

while len(upper_left_loop)>len(dummy_loop)/zoom_factor:
    dummy_upper_left_loop=[upper_left_loop[:,0],upper_left_loop[:,1]]
    centroid2=centroid_calculator(dummy_upper_left_loop)
    dummy_upper_left_loop=upper_left_loop
    dummy_upper_left_loop=PV_Loop_Mask(dummy_upper_left_loop,centroid2[0],'-x')
    dummy_upper_left_loop=PV_Loop_Mask(dummy_upper_left_loop,centroid2[1],'+y')
    if len(dummy_upper_left_loop)>1:
        size_prev=len(upper_left_loop)
        size_new=len(dummy_upper_left_loop)
        if size_prev==size_new:
            break
        else:
            upper_left_loop=dummy_upper_left_loop
            upper_left_centroid=centroid2
            i=i+1
    else:
        break
plt.plot(upper_left_centroid[0], upper_left_centroid[1], linestyle=':', marker='x', color='aqua')
print('The upper left loop was zoomed {} times'.format(i)) 
plt.plot(upper_left_loop[:,0], upper_left_loop[:,1], linestyle=':', marker='.', color='blue') 

if upper_left_pass_fail==1:
    print('Angel')
    derivatives=d_dt(upper_left_loop)

    #Note- making an assumption it is abs value
    #index=np.argmax(abs(derivatives[1]))
    index=np.argmin((derivatives[1]))
    #Save this data to the End Systole_ddt array
    ESddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
    #EndSystole_ddt.append(ESddt)


    index=derivatives[2][index]
    index=int(index)

    hold=dummy_loop[index,:] 

    hold=hold.tolist()
    ESPV_ensemble.append(hold) 
else:
    print('Devil')
    top_right_distances=corner_hunter(upper_left_loop,upper_left_centroid)
    max_dist=np.max(top_right_distances[0])
    point_max_dist=np.argmax(top_right_distances[0])
    index=int(top_right_distances[1][point_max_dist])

    hold=dummy_loop[index,:]
    #Save this data to the End Systole_ddt array
    ESddt=[0,0,hold]
    #EndSystole_ddt.append(ESddt)
    hold=dummy_loop[int(hold[2]),:] 
    hold=hold.tolist()
    ESPV_ensemble.append(hold) 
plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')

###############################################################################################





###############################################################################################    
#Zoom the lower right side
i=0

while len(lower_right_loop)>len(dummy_loop)/zoom_factor:
    dummy_lower_right_loop=[lower_right_loop[:,0],lower_right_loop[:,1]]
    centroid2=centroid_calculator(dummy_lower_right_loop)

    dummy_lower_right_loop=lower_right_loop
    dummy_lower_right_loop=PV_Loop_Mask(dummy_lower_right_loop,centroid2[0],'+x')
    dummy_lower_right_loop=PV_Loop_Mask(dummy_lower_right_loop,centroid2[1],'-y')
    if len(dummy_lower_right_loop)>1:
        size_prev=len(lower_right_loop)
        size_new=len(dummy_lower_right_loop)
        if size_prev==size_new:
            break
        else:
            lower_right_loop=dummy_lower_right_loop
            i=i+1
            lower_right_centroid=centroid2

    else:    
        break
plt.plot(lower_right_centroid[0], lower_right_centroid[1], linestyle=':', marker='x', color='aqua')
print('The lower right loop was zoomed {} times'.format(i)) 
plt.plot(lower_right_loop[:,0], lower_right_loop[:,1], linestyle=':', marker='.', color='blue')


if lower_right_pass_fail==1:
    print('Angel')
    derivatives=d_dt(lower_right_loop)

    #Note- making an assumption it is abs value
    #index=np.argmax(abs(derivatives[1]))
    index=np.argmax((derivatives[1]))
    #Save this data to the End Systole_ddt array
    EDddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
    #EndDiastole_ddt.append(ESddt)


    index=derivatives[2][index]
    index=int(index)

    hold=dummy_loop[index,:] 
    hold=hold.tolist()
    EDPV_ensemble.append(hold) 
else:
    print('Devil')
    bottom_right_distances=corner_hunter(lower_right_loop,centroid)
    max_dist=np.max(bottom_right_distances[0])
    point_max_dist=np.argmax(bottom_right_distances[0])
    index=int(bottom_right_distances[1][point_max_dist])
    hold=dummy_loop[index,:]
    #Save this data to the End Diatole_ddt array
    EDddt=[0,0,hold]
    #EndDiastole_ddt.append(ESddt)

    hold=dummy_loop[int(hold[2]),:] 
    hold=hold.tolist()
    EDPV_ensemble.append(hold) 
plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')

###############################################################################################

plt.show()


In [ ]:
SW_ensemble,SV_ensemble,EJF_ensemble,HR_ensemble,CO_ensemble,Elastance_ensemble,Pmax_ensemble, Pmin_ensemble, Vmax_ensemble, Vmin_ensemble=Calculator(time_step,ensemble_loop,Cycle_Start,len(ensemble_loop),ESPV_ensemble[0],EDPV_ensemble[0])


## Output Collected Data into an Excel Worksheet

In [ ]:
output_file_name=file_name+'_'+sheet_name

titles=['Stroke Work', 'Stroke Volume' , 'Ejection Fraction', 'Heart Rate', 'Cardiac Output', 'Elastance', 'Max Pressure', 'Min Pressure', 'Max Volume', 'Min Volume', 'ESV', 'ESP', 'EDV', 'EDP']

rawmean=np.array([rawmean_SW,rawmean_SV,rawmean_EJF,rawmean_HR,rawmean_CO,rawmean_Elastance,rawmean_MaxPressure,rawmean_MinPressure,rawmean_MaxVolume,rawmean_MinVolume,rawmean_ESV,rawmean_ESP,rawmean_EDV,rawmean_EDP])
Output_rawmean= pd.DataFrame(rawmean)
Output_rawmean=Output_rawmean.T
Output_rawmean.columns=titles


Ensemble_Hemodata=np.array([SW_ensemble,SV_ensemble,EJF_ensemble,HR_ensemble,CO_ensemble,Elastance_ensemble,Pmax_ensemble, Pmin_ensemble, Vmax_ensemble, Vmin_ensemble, ESPV_ensemble[0][0],ESPV_ensemble[0][1],EDPV_ensemble[0][0],EDPV_ensemble[0][1]])
Output_Ensemble_Hemodata= pd.DataFrame(Ensemble_Hemodata)
Output_Ensemble_Hemodata=Output_Ensemble_Hemodata.T
Output_Ensemble_Hemodata.columns=titles
Output_Ensemble_Hemodata= pd.DataFrame(Output_Ensemble_Hemodata)



#pure_raw_data=np.zeros((int(len(StrokeWork)-1),int(len(titles)-1)))

#for step in range (len(StrokeWork))
pure_raw_data=np.vstack([StrokeWork, StrokeVolume, EjectionFraction, Heart_Rate, CardiacOutput, Elastance, MaxPressure, MinPressure, MaxVolume, MinVolume, ESV_total, ESP_total, EDV_total, EDP_total])
#pure_raw_data=np.vstack([StrokeWork, StrokeVolume, EjectionFraction, Heart_Rate, CardiacOutput, Elastance, MaxPressure, MinPressure, MaxVolume, MinVolume])
pure_raw_data= pd.DataFrame(pure_raw_data)
pure_raw_data=pure_raw_data.T
#titles2=['Stroke Work', 'Stroke Volume' , 'Ejection Fraction', 'Heart Rate', 'Cardiac Output', 'Elastance', 'Max Pressure', 'Min Pressure', 'Max Volume', 'Min Volume']
pure_raw_data.columns=titles
pure_raw_data= pd.DataFrame(pure_raw_data)


Ensemble_Frame=[ensemble_pressure,ensemble_volume]
Ensemble_Frame=np.array(Ensemble_Frame)
Ensemble_Frame=np.transpose(Ensemble_Frame)
Ensemble_Frame= pd.DataFrame(Ensemble_Frame, columns = ['Pressure', 'Volume'] )


with pd.ExcelWriter("PVLoop_Output_{}.xlsx".format(output_file_name)) as writer:
    Output_rawmean.to_excel(writer, sheet_name="Raw Mean Data", index=0)
    Output_Ensemble_Hemodata.to_excel(writer, sheet_name="Ensemble Data", index=0)
    pure_raw_data.to_excel(writer, sheet_name="All Raw Data", index=0)
    Ensemble_Frame.to_excel(writer, sheet_name="Ensemble PV Loop", index=0)

# Super-impose Ensemble Average onto the Raw Data and Save Image

In [ ]:

#Compare loops


fig = plt.figure()
fig.patch.set_facecolor('blue')
fig.patch.set_alpha(0.05)
ax = fig.add_subplot(111)


final_number_of_loops=len(Physiological_Loops)

#plt.plot(vol, press, linestyle=':', marker='*', color='red')
#plt.plot(ensemble_volume, ensemble_pressure, linestyle='-', marker='.', color='green')

plt.plot(vol, press, color='violet')
plt.plot(ensemble_volume, ensemble_pressure, color='green')

plt.xlabel('Volume', color='black')
plt.ylabel('Pressure', color='black')
plt.title('Pressure vs Volume of All Loops'.format(final_number_of_loops), color='black',)

ax.legend(['Raw Data', 'Ensemble Average'])

plt.plot()
plt.show()

In [ ]:

#Compare loops


fig = plt.figure()
fig.patch.set_facecolor('blue')
fig.patch.set_alpha(0.05)
ax = fig.add_subplot(111)


final_number_of_loops=len(Physiological_Loops)

#plt.plot(vol, press, linestyle=':', marker='*', color='red')
#plt.plot(ensemble_volume, ensemble_pressure, linestyle='-', marker='.', color='green')

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
    

plt.plot(ensemble_volume, ensemble_pressure, color='green')

plt.xlabel('Volume', color='black')
plt.ylabel('Pressure', color='black')
plt.title('CLEANED Pressure vs Volume of {} Loops'.format(final_number_of_loops), color='green',)

#ax.legend(['Raw Data', 'Ensemble Average'])

plt.plot()
plt.show()

#Save Image <br>
#should be moved a cell up <br>
plt.savefig('PVLoop_Ensemble_{}.png'.format(output_file_name))

In [ ]:
if sheet_name=='T0.1' or sheet_name=='T30.1' or sheet_name=='T74.1':
    print("Let's hunt some ESPVR and EDPVR!")
else:
    exit
    import sys 
    sys.exit()

### Plot trends in (ESV, ESP) and (EDV, EDP) points

In [ ]:
plt.plot(ESP_total)
plt.plot(ESV_total)
plt.legend(['ESP', 'ESV'])
plt.title('ESV/P vs Loop Number')
plt.show()

plt.plot(EDP_total)
plt.plot(EDV_total)
plt.legend(['EDP', 'EDV'])
plt.title('EDV/P vs Loop Number')
plt.show()

plt.plot(ESV_total)
plt.plot(EDV_total)
plt.legend(['ESV', 'EDV'])
plt.title('ESV/EDV vs Loop Number')
plt.show()

plt.plot(ESP_total)
plt.plot(EDP_total)
plt.legend(['ESP', 'EDP'])
plt.title('ESP/EDP vs Loop Number')
plt.show()

## ESPVR Calculation

In [ ]:
plt.scatter(ESV_total,ESP_total)
plt.grid(True)
plt.xlabel("ESV")
plt.ylabel("ESP")
plt.show()

In [ ]:
#Change into numpy arrays for this
ESV_total_array=np.array(ESV_total)
ESP_total_array=np.array(ESP_total)

##### Using Machine Learning Model to Obtain the Linear Regression

In [ ]:
regr = linear_model.LinearRegression()
regr.fit(ESV_total_array.reshape(-1,1), ESP_total_array)

In [ ]:
print('Intercept: \n', regr.intercept_)
print('Coefficients: \n', regr.coef_[0])

##### Using statsmodels

In [ ]:
ESV_total_array = sm.add_constant(ESV_total_array) # adding a constant
 
model = sm.OLS(ESP_total_array, ESV_total_array).fit()
predictions = model.predict(ESV_total_array)
 
print_model = model.summary()
print(print_model)

#### Linear Representation

In [ ]:
#find line of best fit for ESPVR
for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
    
plt.plot(ensemble_volume, ensemble_pressure, color='green')

E_es, P0 = np.polyfit(ESV_total, ESP_total, 1)
print('Slope')
print(E_es)
print('y-int')
print(P0)
ESVPR_volume_array=np.linspace(min(ESV_total)*.7,np.max(ESV_total), 100)

ESVPR=E_es*ESVPR_volume_array+P0

plt.plot(ESVPR_volume_array, ESVPR)

#slope, intercept = np.polyfit(ESVPR_volume_array, ESVPR, 1)

plt.title('ESPVR Plot')
plt.xlabel('Volume')
plt.ylabel('Pressure')
plt.show()


#### Non-linear representation

In [ ]:
V0_estimate=-P0/E_es

def ESPVR12(V, a1, a0):
    return a1 * V +a0

def ESPVR34(V, a2, a1, a0):
    return a2 * V**2 + a1 * V + a0

def ESPVR56 (V, a, B, V0_estimate):
    return 1 / (a + B * V) * np.log (V/V0_estimate)


In [ ]:
#Hypothetical Range of Values for the Volume
volume_line=np.linspace(V0_estimate,np.max(ESV_total))

In [ ]:
#Linear 2
poptS2=[]
pcovS2=[]
perrS2=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')

poptS2, pcovS2 = curve_fit(ESPVR12, ESV_total, ESP_total)

perrS2 = np.sqrt(np.diag(pcovS2))
print(perrS2)

ESPVR2=ESPVR12(volume_line,*poptS2)
plt.plot(ESV_total,ESP_total, 'o', volume_line,ESPVR2)
plt.show()
print('The V0 value here is {}'.format(V0_estimate))

In [ ]:
#Quad 2
poptS4=[]
pcovS4=[]
perrS4=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')
poptS4, pcovS4 = curve_fit(ESPVR34, ESV_total, ESP_total)

perrS4 = np.sqrt(np.diag(pcovS4))
print(perrS4)

ESPVR4=ESPVR34(volume_line,*poptS4)
plt.plot(ESV_total,ESP_total, 'o', volume_line,ESPVR4)
plt.show()

In [ ]:
#Log 2

poptS6=[]
pcovS6=[]
perrS6=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')

poptS6, pcovS6 = curve_fit(ESPVR56, ESV_total, ESP_total, maxfev=100000)

perrS6 = np.sqrt(np.diag(pcovS6))
print(perrS6)


ESPVR6=ESPVR56(volume_line,*poptS6)
plt.plot(ESV_total,ESP_total, 'o', volume_line,ESPVR6)
plt.show()

In [ ]:
#Add the point P(V0,Lin)=0
ESV_total.insert(0, V0_estimate)
ESP_total.insert(0, 0)

In [ ]:
#Linear 1

poptS1=[]
pcovS1=[]
perrS1=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')
poptS1, pcovS1 = curve_fit(ESPVR12, ESV_total, ESP_total)

perrS1 = np.sqrt(np.diag(pcovS1))
print(perrS1)

ESPVR1=ESPVR12(volume_line,*poptS1)
plt.plot(ESV_total,ESP_total, 'o', volume_line,ESPVR1)
plt.show()

In [ ]:
#Quad 1

poptS3=[]
pcovS3=[]
perrS3=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')
poptS3, pcovS3 = curve_fit(ESPVR34, ESV_total, ESP_total)

perrS3 = np.sqrt(np.diag(pcovS3))
print(perrS3)

ESPVR3=ESPVR34(volume_line,*poptS3)
plt.plot(ESV_total,ESP_total, 'o', volume_line,ESPVR3)
plt.show()

In [ ]:
#Log 1

poptS5=[]
pcovS5=[]
perrS5=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')

poptS5, pcovS5 = curve_fit(ESPVR56, ESV_total, ESP_total, maxfev=1000000)

perrS5 = np.sqrt(np.diag(pcovS5))
print(perrS5)

ESPVR5=ESPVR56(volume_line,*poptS5)
plt.plot(ESV_total,ESP_total, 'o', volume_line,ESPVR5)
plt.show()


In [ ]:
titles=['Linear-Regression y-intercept', 'Linear-Regression Coefficients','Poly-Fit y-intercept', 'Poly-Fit Coefficients']

EPSVR_data=np.array([regr.intercept_,regr.coef_[0],P0,E_es], dtype=object)
EPSVR_data= pd.DataFrame(EPSVR_data)
EPSVR_data=EPSVR_data.T
EPSVR_data.columns=titles


titles=['poptS1', 'pcovS1','perrS1','poptS2', 'pcovS2', 'perrS2', 'poptS3', 'pcovS3','perrS3', 'poptS4' , 'pcovS4','perrS4', 'poptS5', 'pcovS5','perrS5', 'poptS6', 'pcovS6', 'perrS6']
ESPVR_nonlin_data=np.array([poptS1, pcovS1,perrS1,poptS2, pcovS2,perrS2, poptS3, pcovS3,perrS3, poptS4, pcovS4, perrS4, poptS5, pcovS5, perrS5, poptS6, pcovS6,perrS6], dtype=object)
ESPVR_nonlin_data= pd.DataFrame(ESPVR_nonlin_data)
ESPVR_nonlin_data=ESPVR_nonlin_data
ESPVR_nonlin_data.index=titles

with pd.ExcelWriter("PVLoop_ESPVR_{}.xlsx".format(output_file_name)) as writer:
    EPSVR_data.to_excel(writer, sheet_name="Linear ESPVR", index=0)
    ESPVR_nonlin_data.to_excel(writer, sheet_name="Non-Linear ESPVR", index=1)
  

####### https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html

## EDPVR Calculation

![EDPVR_FIts.jpg](attachment:EDPVR_FIts.jpg)

In [ ]:
plt.scatter(EDV_total,EDP_total)
plt.grid(True)
plt.xlabel("EDV")
plt.ylabel("EDP")
plt.show()

In [ ]:
def EDPVR1(V, A, B, alpha):
    return A + B * np.exp(alpha * V)

def EDPVR2(V, C, beta):
    return C * np.exp(beta * V)

def EDPVR3(V, D, a):
    return D + a * V**3

def EDPVR4(V, a0, a1, a2, a3):
    return a0 + a1 * V + a2 * V**2+ a3 * V**3

def EDPVR5(V, b, c, gamma):
    return b+ c * V**gamma

def EDPVR6(V, d, delta):
    return d* V**delta

In [ ]:
#Hypothetical Range of Values for the Volume
volume_line=np.linspace(V0_estimate,np.max(EDV_total))

In [ ]:
#Exponential 1

popt1=[]
pcov1=[]
perr1=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')
popt1, pcov1 = curve_fit(EDPVR1, EDV_total, EDP_total,maxfev=1000000)

perr1 = np.sqrt(np.diag(pcov1))
print(perr1)

EDPVR1=EDPVR1(volume_line,*popt1)
plt.plot(EDV_total,EDP_total, 'o', volume_line,EDPVR1)
plt.show()

In [ ]:
#Exponential 2

popt2=[]
pcov2=[]
perr2=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')
popt2, pcov2 = curve_fit(EDPVR2, EDV_total, EDP_total,maxfev=1000000)



perr2 = np.sqrt(np.diag(pcov2))
print(perr2)


EDPVR2=EDPVR2(volume_line,*popt2)
plt.plot(EDV_total,EDP_total, 'o', volume_line,EDPVR2)
plt.show()

In [ ]:
#Cubic 1

popt3=[]
pcov3=[]
perr3=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')
popt3, pcov3 = curve_fit(EDPVR3, EDV_total, EDP_total,maxfev=1000000)


perr3 = np.sqrt(np.diag(pcov3))
print(perr3)

EDPVR3=EDPVR3(volume_line,*popt3)
plt.plot(EDV_total,EDP_total, 'o', volume_line,EDPVR3)
plt.show()

In [ ]:
#Cubic 2

popt4=[]
pcov4=[]
perr4=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')

popt4, pcov4 = curve_fit(EDPVR4, EDV_total, EDP_total,maxfev=1000000)


perr4 = np.sqrt(np.diag(pcov4))
print(perr4)


EDPVR4=EDPVR4(volume_line,*popt4)
plt.plot(EDV_total,EDP_total, 'o', volume_line,EDPVR4)
plt.show()

In [ ]:
#Power 1

popt5=[]
pcov5=[]
perr5=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
popt5, pcov5 = curve_fit(EDPVR5, EDV_total, EDP_total, maxfev=10000)


perr5 = np.sqrt(np.diag(pcov5))
print(perr5)

EDPVR5=EDPVR5(volume_line,*popt5)
plt.plot(EDV_total,EDP_total, 'o', volume_line,EDPVR5)
plt.show()

In [ ]:
#Power 2

popt6=[]
pcov6=[]
perr6=[]

for i in range(len(Physiological_Loops)):
    dummy_vol=Physiological_Loops[i][0]
    dummy_press=Physiological_Loops[i][1]
    plt.plot(dummy_vol, dummy_press, color='red')
plt.plot(ensemble_volume, ensemble_pressure, color='green')
popt6, pcov6 = curve_fit(EDPVR6, EDV_total, EDP_total, maxfev=1000000)

perr6 = np.sqrt(np.diag(pcov6))
print(perr6)

EDPVR6=EDPVR6(volume_line,*popt6)
plt.plot(EDV_total,EDP_total, 'o', volume_line,EDPVR6)
plt.show()

## Documentation for poptarray -pending fixes
poptarray
Optimal values for the parameters so that the sum of the squared residuals of f(xdata, *popt) - ydata is minimized.

pcov2-D array
The estimated covariance of popt. The diagonals provide the variance of the parameter estimate. To compute one standard deviation errors on the parameters use perr = np.sqrt(np.diag(pcov)).

How the sigma parameter affects the estimated covariance depends on absolute_sigma argument, as described above.

If the Jacobian matrix at the solution doesn’t have a full rank, then ‘lm’ method returns a matrix filled with np.inf, on the other hand ‘trf’ and ‘dogbox’ methods use Moore-Penrose pseudoinverse to compute the covariance matrix.

plt.xlabel('x')
plt.ylabel('y')
#plt.legend()

plt.plot(ESV_total, EDPVR1(ESV_total, *popt), 'r-',
         label='fit: a=%5.3f, b=%5.3f, c=%5.3f' % tuple(popt))

#plt.plot(ESV_total, ESP_total)
plt.show()

print(popt)
print(pcov)

ESV_total
ESP_total
EDV_total
EDP_total

# Compare Loops with Stonko
##### only applies if you have Stonko's data on your sheet.

print(max_vol)
print(min_vol)

print(max_press)
print(min_press)


print(sheet_name)
stonko_sheet='Stonko '+sheet_name
Stonko_Ensemble = pd.read_excel (file_name, stonko_sheet)
Stonko_Ensemble=Stonko_Ensemble.to_numpy()
print('Green=Fahim, Blue=Stonko, Red=Raw')

fig = plt.figure()
fig.patch.set_facecolor('pink')
fig.patch.set_alpha(0.9)
ax = fig.add_subplot(111)


final_number_of_loops=len(Physiological_Loops)

plt.plot(vol, press, linestyle=':', marker='*', color='red')
plt.plot(ensemble_volume, ensemble_pressure, linestyle='-', marker='.', color='green')
plt.plot(Stonko_Ensemble[:,0],Stonko_Ensemble[:,1], linestyle='-', marker='.', color='blue')
plt.xlabel('Volume', color='green')
plt.ylabel('Pressure', color='blue')
plt.title('Pressure vs Volume of {} Loops'.format(final_number_of_loops), color='red')
#Green is mine
#Blue is Stonko's

ax.legend(['Raw Data', 'Fahim Ensemble', 'Stonko'], loc='upper left')

plt.plot()
plt.savefig('PVLoop_Ensemble_{}.png'.format(output_file_name))


### Fully Mined Data outputs- just need to organize into my outputs
ESP, EDP, ESV,EDV,Pmax,Pmin, dPmax,dPmin, Vmax, Vmin,
<br> 
#  The above have all been mined !
<br>

# Reassessing
P@dP/dt max,P@dV/dt max, P@dP/dt min,P@dV/dt min

# Need to learn how to mine-
max power, lPower, PVA, PE, Eff, Tau Weiss, Tau Logistics, Tau Glantz,
Tau Mirsky

def func(x, a, b, c):
    return a * np.exp(-b * x) + c

xdata = np.linspace(0, 4, 50)
y = func(xdata, 2.5, 1.3, 0.5)
rng = np.random.default_rng()
y_noise = 0.2 * rng.normal(size=xdata.size)
ydata = y + y_noise
plt.plot(xdata, ydata, 'b-', label='data')

popt, pcov = curve_fit(func, xdata, ydata)
popt

plt.plot(xdata, func(xdata, *popt), 'r-',
         label='fit: a=%5.3f, b=%5.3f, c=%5.3f' % tuple(popt))
popt, pcov = curve_fit(func, xdata, ydata, bounds=(0, [3., 1., 0.5]))
popt
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
titles=['popt1', 'pcov1','perr1','popt2', 'pcov2', 'perr2', 'popt3', 'pcov3','perr3', 'popt4' , 'pcov4','perr4', 'popt5', 'pcov5','perr5', 'popt6', 'pcov6', 'perr6']
EDPVR_data=np.array([popt1, pcov1,perr1,popt2, pcov2,perr2, popt3, pcov3,perr3, popt4, pcov4, perr4, popt5, pcov5, perr5, popt6, pcov6,perr6], dtype=object)

EDPVR_data= pd.DataFrame(EDPVR_data)
EDPVR_data=EDPVR_data
EDPVR_data.index=titles
#EDPVR_data.columns=titles


with pd.ExcelWriter("PVLoop_EDPVR_{}.xlsx".format(output_file_name)) as writer:
    EDPVR_data.to_excel(writer, sheet_name="EDPVR", index=1)